In [1]:
import os 
import numpy as np
import pandas as pd
import scipy
import sklearn.metrics as skmetrics

# plotting
import matplotlib.pyplot as plt
import seaborn as sns

# Pytorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import lightning as L

import torchmetrics
from torchmetrics.regression import PearsonCorrCoef

In [ ]:
# the dataloaders load the tensors from memory one by one, could potentially become a bottleneck

class ProtEmbeddingDataset(Dataset):
    """
    Dataset for the embeddings of the mutated sequences
    You can the get_item() method to return the data in the format you want
    """
    def __init__(self, tensor_folder, csv_file, id_col="name", label_col="ddG_ML"):
        """
        Initialize the dataset
        input at init: 
            tensor_folder: path to the directory with the embeddings we want to use, eg. "/home/data/mega_train_embeddings"
            cvs_file: path to the csv file corresponding to the data, eg. "home/data/mega_train.csv"
        """
        self.tensor_folder = tensor_folder
        self.df = pd.read_csv(csv_file, sep=",")
        # only use the mutation rows
        self.df = self.df[self.df.mut_type!="wt"]
        # get the labels and ids
        self.labels = torch.tensor(self.df[label_col].values)
        self.ids = self.df[id_col].values
        self.wt_names = self.df["WT_name"].values

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        # load embeddings
        # mutation embedding
        tensor_path = os.path.join(self.tensor_folder, self.ids[idx] + ".pt")
        tensor = torch.load(tensor_path)['mean_representations'][6]

        # wildtype embedding, uncomment if you want to use this, too
        """
        tensor_wt = None
        try:
            # try loading with _wte.pt suffix
            tensor_path_wt = os.path.join(self.tensor_folder, self.wt_names[idx] + "_wte.pt")
            tensor_wt = torch.load(tensor_path_wt)['mean_representations'][6]
        except FileNotFoundError:
            try:
                # try loading with _wty.pt suffix
                tensor_path_wt = os.path.join(self.tensor_folder, self.wt_names[idx] + "_wty.pt")
                tensor_wt = torch.load(tensor_path_wt)['mean_representations'][6]
            except FileNotFoundError:
                print("No file found for WT embedding of ", self.wt_names[idx], " with either _wte.pt or _wty.pt suffix.")
                pass  # neither file found, tensor_wt remains None
        """

        label = self.labels[idx] # ddG value
        # returns a tuple of the input embedding and the target ddG values
        return tensor, label.float()

In [ ]:
# usage 
# make sure to adjust the paths to where your files are located
dataset_train = ProtEmbeddingDataset('project_data/mega_train_embeddings', 'project_data/mega_train.csv')
dataset_val = ProtEmbeddingDataset('project_data/mega_val_embeddings', 'project_data/mega_val.csv')
# dataset_test = ProtEmbeddingDataset('project_data/mega_test_embeddings', 'project_data/mega_test.csv') # only once available

dataloader_train = DataLoader(dataset_train, batch_size=1024, shuffle=True, num_workers=16)
dataloader_val = DataLoader(dataset_val, batch_size=512, shuffle=False, num_workers=16)
# dataloader_test = DataLoader(dataset_test, batch_size=32, shuffle=False)